# Clustering with Multiple Features

# Introduction

In the previous lesson you built a K-Means model using only two features. That
kept everything visible on a single scatter plot — perfect for intuition, but a
luxury real segmentation rarely affords. Households differ along *many* financial
axes at once, and a serious segmentation has to weigh all of them together.

This lesson scales clustering up. You'll select informative features from
hundreds of candidates using **variance**, guard that selection against outliers
with **trimmed variance**, put a `StandardScaler` in front of K-Means so no single
dollar amount dominates, and — since you can't plot five dimensions directly — use
**Principal Component Analysis (PCA)** to compress the result into a picture.

🎯 **By the end of this notebook you will be able to:**

-   Calculate and compare variance across features to identify informative
    variables for clustering.
-   Apply trimmed variance to handle outliers when selecting features.
-   Build a K-Means clustering pipeline that includes feature standardization.
-   Use inertia and silhouette scores to select an appropriate number of
    clusters.
-   Apply PCA to reduce high-dimensional data for visualization.
-   Interpret and communicate multi-feature cluster results.

➡️ The arc: pick good features → put them on a common scale → choose *k* → fit →
and finally *see* the clusters through PCA.

## Watch first

The video previews how multi-feature segmentation differs from the two-feature
case. The recurring theme: more features mean more information, but also more
ways for scale and outliers to mislead you — which is exactly what the techniques
below defend against.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1105512918", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

Last lesson you clustered on two features and could read the result straight off
a scatter plot. But the Survey of Consumer Finances carries *hundreds* of columns
describing household finances. Restricting yourself to two of them throws away
most of what the survey knows.

Here you'll build a richer segmentation: select five high-variance features and
cluster households on all of them at once. The catch is dimensionality — beyond
two features you can no longer draw the clusters directly. **PCA** will be your
answer, projecting the five-dimensional result down to two axes you can plot while
keeping as much of the structure as possible.

📌 Your final deliverable is a K-Means model that segments credit-constrained
households (turned down for credit or feared denial) with net worth under
\$2 million into meaningful financial groups.

## Feature selection using variance

With many features to choose from, one principled starting point is to keep the
ones with the highest **variance**. The logic: a feature that barely changes from
household to household can't help tell households apart, while a high-variance
feature carries real signal about how they differ.

⚠️ But raw variance has a blind spot — **outliers**. A single ultra-wealthy
household can single-handedly inflate the variance of any wealth feature, making
it look informative when it's really just reflecting one extreme value. The fix is
**trimmed variance**: compute variance *after* discarding the most extreme
observations (here, the top and bottom 10%), so the measure reflects typical
households rather than the rare giant.

### Demonstrating variance calculations

In [ ]:
import pandas as pd
from scipy.stats.mstats import trimmed_var

# Create toy data with an outlier
toy_df = pd.DataFrame({
    "income": [50000, 55000, 60000, 52000, 1000000],  # Last value is outlier
    "age": [30, 35, 40, 32, 38]
})

# Regular variance (sensitive to outliers)
regular_var = toy_df.var()
print("Regular variance:")
print(regular_var)

In [ ]:
# Trimmed variance removes top and bottom 10% before calculating
# limits=(0.1, 0.1) trims 10% from each tail
trimmed_var_result = toy_df.apply(trimmed_var, limits=(0.1, 0.1))
print("\nTrimmed variance (10% from each tail):")
print(trimmed_var_result)

🔍 Look at the contrast: the income feature's *regular* variance is enormous —
driven almost entirely by the single \$1,000,000 outlier — while its *trimmed*
variance is far smaller and reflects the spread among ordinary households. That
gap is precisely why we trim before ranking features.

📦 **Key points**

-   [`scipy.stats.mstats.trimmed_var`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.trimmed_var.html):
    Computes variance after trimming a proportion from each tail.
-   [`pandas.DataFrame.var`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.var.html):
    Returns variance for each column.
-   [`pandas.DataFrame.apply`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html):
    Applies a function along an axis of the DataFrame.

## Why standardization matters for multi-feature clustering

K-Means measures similarity with **Euclidean distance**, and distance is blind to
meaning — it only sees magnitude. If one feature ranges 0 → 1,000,000 (like
`ASSET`) and another ranges 0 → 10 (like vehicle count), the million-scale feature
swamps the distance calculation and the small-scale feature is effectively
ignored, no matter how informative it is.

**Standardization** levels the playing field by rescaling every feature to mean 0
and standard deviation 1:

$$z = \frac{x - \mu}{\sigma}$$

After this transform, "one unit" means "one standard deviation" for *every*
feature, so each contributes to the distance on equal footing.

### Demonstrating StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Create toy data with different scales
X_toy = pd.DataFrame({
    "assets": [100000, 200000, 150000, 180000],
    "vehicles": [1, 2, 1, 3]
})

print("Before standardization:")
print(X_toy.agg(["mean", "std"]).round(2))

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_toy)

# Convert back to DataFrame for inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=X_toy.columns)
print("\nAfter standardization:")
print(X_scaled_df.agg(["mean", "std"]).round(2))

✅ Check the after-table: both features now sit at mean ≈ 0 and standard deviation
≈ 1. The raw values are gone, replaced by *how many standard deviations from
average* each value is — the common currency K-Means needs.

📦 **Key points**

-   [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html):
    Standardizes features by removing mean and scaling to unit variance.

## Building pipelines for reproducible workflows

There's a subtle trap in standardizing by hand: you must remember to scale every
dataset the same way, in the same order, every time. A **pipeline** removes that
burden by chaining the scaler and the model into one object — call `.fit()` once
and scaling always happens first, automatically.

### Demonstrating make_pipeline

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans

# Create a pipeline that standardizes then clusters
model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=2, random_state=42, n_init=10)
)

# Fit the pipeline on toy data
model.fit(X_toy)

# Access the KMeans step to get labels
print("Cluster labels:", model[-1].labels_)
print("Inertia:", model[-1].inertia_)

🔧 The pipeline standardizes the data before handing it to K-Means, with no manual
bookkeeping. To reach inside, index the steps — `model[-1]` grabs the last step
(KMeans) — or use the named form `model.named_steps["kmeans"]`. You'll use both
styles in the exercises.

📦 **Key points**

-   [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html):
    Creates a pipeline from a sequence of estimators.

## Selecting the number of clusters

More features make the choice of *k* more consequential, so you'll lean on the
same two complementary metrics as last lesson — now read together:

| Metric | Question it answers | Direction | Caveat |
| --- | --- | --- | --- |
| **Inertia** (elbow method) | Are points tight around their centroids? | Lower is tighter | Always falls as *k* grows — read the *bend*, not the minimum |
| **Silhouette score** | Are clusters distinct from each other? | Higher (range −1 → 1) | Can fall when *k* is too high — flags over-segmentation |

🔍 Inertia alone can't tell you when you have *too many* clusters — it keeps
dropping. Silhouette can. Used together, the inertia elbow proposes a *k* and the
silhouette score confirms whether that *k* yields genuinely separated groups.

### Demonstrating cluster evaluation metrics

In [ ]:
from sklearn.metrics import silhouette_score

# Generate slightly larger toy data for meaningful metrics
np.random.seed(42)
X_eval = pd.DataFrame({
    "feature1": np.concatenate([
        np.random.normal(0, 1, 20),
        np.random.normal(5, 1, 20)
    ]),
    "feature2": np.concatenate([
        np.random.normal(0, 1, 20),
        np.random.normal(5, 1, 20)
    ])
})

# Fit a pipeline and extract metrics
eval_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=2, random_state=42, n_init=10)
)
eval_model.fit(X_eval)

# Get inertia from the KMeans step
inertia = eval_model[-1].inertia_
print(f"Inertia: {inertia:.2f}")

# Calculate silhouette score (needs original data and labels)
sil_score = silhouette_score(X_eval, eval_model[-1].labels_)
print(f"Silhouette score: {sil_score:.2f}")

The standard workflow: fit models across a range of *k* (say 2 to 12), record both
metrics at each step, and plot them — exactly what you'll do on the real data in
Section 6.

📦 **Key points**

-   [`sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html):
    K-Means clustering algorithm. Key attributes: `inertia_`, `labels_`.
-   [`sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html):
    Computes the mean silhouette coefficient for all samples.

## Visualizing high-dimensional clusters with PCA

Here's the bind: cluster on five features and there's no scatter plot that shows
them all. **Principal Component Analysis (PCA)** resolves it by projecting the
high-dimensional data onto a few new axes that capture as much variation as
possible.

🧱 These new axes — **principal components** — are weighted blends of your original
features. PC1 is the single direction along which the data varies most; PC2 is the
next-most-variable direction perpendicular to it; and so on. Plotting PC1 vs. PC2
gives you the most faithful 2D shadow of a 5D cloud.

### Demonstrating PCA

In [ ]:
from sklearn.decomposition import PCA

# Create toy data with 4 features
np.random.seed(42)
X_high_dim = pd.DataFrame({
    "f1": np.random.randn(50),
    "f2": np.random.randn(50),
    "f3": np.random.randn(50),
    "f4": np.random.randn(50)
})

print(f"Original shape: {X_high_dim.shape}")

# Reduce to 2 dimensions
pca = PCA(n_components=2, random_state=42)
X_reduced = pca.fit_transform(X_high_dim)

print(f"Reduced shape: {X_reduced.shape}")
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

In [ ]:
# Convert to DataFrame for easier handling
X_pca_df = pd.DataFrame(X_reduced, columns=["PC1", "PC2"])
X_pca_df.head()

📊 After transforming, you get a 2D table you can scatter-plot, colored by cluster.
The trade-off: PC1 and PC2 are *combinations* of the original features, so the
axes no longer mean "dollars of debt" or "number of vehicles." You gain a picture
of cluster separation at the cost of directly interpretable axes.

📦 **Key points**

-   [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html):
    Principal Component Analysis for dimensionality reduction.

## Plotting recap with Plotly Express

This lesson swaps Matplotlib/Seaborn for **Plotly Express**, which renders
interactive charts. Here's a quick tour of the three chart types you'll use.

### Bar charts for comparing values

In [ ]:
import plotly.express as px

# Example: horizontal bar chart
categories = pd.Series([300, 500, 200], index=["A", "B", "C"])
fig = px.bar(
    x=categories.values,
    y=categories.index,
    orientation="h",
    title="Example Horizontal Bar Chart"
)
fig.update_layout(xaxis_title="Value", yaxis_title="Category")
fig.show()

### Scatter plots for cluster visualization

In [ ]:
# Example: scatter plot with color grouping
scatter_df = pd.DataFrame({
    "x": [1, 2, 3, 4, 5, 6],
    "y": [2, 3, 1, 5, 4, 6],
    "group": ["A", "A", "A", "B", "B", "B"]
})
fig = px.scatter(
    scatter_df,
    x="x",
    y="y",
    color="group",
    title="Example Scatter Plot with Groups"
)
fig.show()

### Box plots for distribution analysis

In [ ]:
# Example: box plot to check for outliers
box_data = pd.DataFrame({"values": [10, 12, 11, 13, 100, 14, 15]})
fig = px.box(box_data, x="values", title="Example Box Plot")
fig.show()

🔍 That box plot is more than decoration — it's how you *spot* the outliers that
break raw variance. The whiskers and far-flung points make extreme values
obvious, motivating the trimmed-variance step you'll apply to the real data.

📦 **Key points**

-   [`plotly.express.bar`](https://plotly.com/python-api-reference/generated/plotly.express.bar):
    Creates bar charts.
-   [`plotly.express.scatter`](https://plotly.com/python-api-reference/generated/plotly.express.scatter):
    Creates scatter plots.
-   [`plotly.express.box`](https://plotly.com/python-api-reference/generated/plotly.express.box):
    Creates box plots.

⚠️ **Common pitfalls and debugging tips**

-   **Forgetting to standardize** — if clusters seem dominated by one feature,
    confirm `StandardScaler` is in your pipeline.
-   **Outliers distorting variance** — if your high-variance features are all
    wealth-related, switch to trimmed variance.
-   **Inconsistent random states** — set `random_state=42` in both `KMeans` and
    `PCA` for reproducible results.
-   **Accessing pipeline components** — `model[-1]` for the last step (KMeans),
    or `model.named_steps["kmeans"]` for named access.
-   **PCA on unscaled data** — standardize before PCA for best results.
-   **Wrong `silhouette_score` input** — pass the original feature matrix `X`
    (not the scaled version) together with the labels from the fitted model.

# Applied Exercises

## 2. Setup

🔧 Collect every import in a single cell at the top so all the names you need —
pandas, Plotly, the scaler, the model, the metric, PCA — are in scope before any
exercise runs.

**Code 6.3.2.1**:

In [ ]:
import pandas as pd
import plotly.express as px
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 3. Data Preparation

### Problem

Before building a multi-feature clustering model, you need to load and filter the
SCF data to a relevant subset. As before, you'll focus on credit-constrained
households — but this time you'll *also* drop ultra-wealthy households so a few
extreme outliers don't distort the analysis.

### Approach

Write a `wrangle` function that loads the data and applies two filters: households
turned down for credit or fearing denial in the past 5 years (`TURNFEAR == 1`),
and households with net worth under \$2 million. Together these carve out a more
homogeneous population to cluster.

📌 As in L2, `wrangle` lives in this notebook — P6 defines it per-lesson rather
than importing from a shared module.

### Tasks

Define `wrangle` to load a CSV and filter for credit-constrained households with
net worth below \$2 million, returning the filtered DataFrame.

**Code 6.3.3.1**:

In [ ]:
def wrangle(filepath):
    # Load CSV file
    df = pd.read_csv(filepath)
    # Create mask for credit-constrained households with NW < $2M
    mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
    # Apply mask
    df = df[mask]
    return df

Now run `wrangle` and verify the result. Expect a DataFrame with over 1,000 rows
and more than 300 columns — the full feature set, restricted to your target
population.

**Code 6.3.3.2**:

In [ ]:
df = wrangle("data/SCFP2019.csv.gz")
print("DataFrame shape:", df.shape)
df.head()

### Checkpoint

🧪 These asserts pin down both filters at once: enough rows survived (> 1,000), the
full column set is present (> 300), `TURNFEAR` contains only the value 1, and the
maximum net worth really is below \$2 million. A failure here points straight at
the mask in `wrangle`.

In [ ]:
assert df.shape[0] > 1000, (
    f"Expected more than 1000 rows, got {df.shape[0]}. "
    "Check your TURNFEAR and NETWORTH filters."
)
assert df.shape[1] > 300, (
    f"Expected more than 300 columns, got {df.shape[1]}. "
    "Make sure you're loading the full dataset."
)
assert df["TURNFEAR"].unique().tolist() == [1], (
    "TURNFEAR should only contain value 1 after filtering."
)
assert df["NETWORTH"].max() < 2e6, (
    f"Max NETWORTH is {df['NETWORTH'].max():,.0f}, expected < 2,000,000."
)
print("Data preparation checkpoint passed!")

## 4. Feature Selection Using Variance

### Problem

The dataset has over 350 features, but most are useless for clustering — a feature
that barely varies can't separate household types. You need to find the columns
that actually capture variation.

### Approach

Calculate variance for every feature, then recompute with trimmed variance to
neutralize outliers, and keep the top 5 by trimmed variance for clustering. Along
the way you'll compare the two rankings with horizontal bar charts to *see* how
much outliers were distorting the picture.

### Tasks

Calculate the variance of every column and pull out the 10 highest. You'll get a
Series indexed by feature name.

**Code 6.3.4.1**:

In [ ]:
# Calculate variance and get top 10
top_ten_var = df.var().sort_values().tail(10)
top_ten_var

Visualize those top 10 high-variance features as a horizontal bar chart to see
which columns have the most spread.

**Code 6.3.4.2**:

In [ ]:
# Create horizontal bar chart
fig = px.bar(
    x=top_ten_var,
    y=top_ten_var.index,
    orientation="h",
    title="SCF: High Variance Features"
)
fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
fig.show()

📊 Notice that the leaders are almost all wealth/asset features with huge raw
variances. Before trusting this ranking, you should ask whether a handful of
extreme households are inflating it — which the next box plot investigates.

Now box-plot the `NHNFIN` feature to inspect its distribution and reveal whether
outliers are inflating its variance.

**Code 6.3.4.3**:

In [ ]:
# Create boxplot to examine distribution
fig = px.box(
    data_frame=df,
    x="NHNFIN",
    title="Distribution of Non-home, Non-Financial Assets"
)
fig.update_layout(xaxis_title="Value [$]")
fig.show()

🔍 The box plot shows extreme right skew: even after filtering to net worth under
\$2 million, a few households sit far above the rest. Those extremes are exactly
what raw variance over-rewards — and the reason the next step switches to **trimmed
variance** for a more robust ranking.

Calculate trimmed variance for all features (trim 10% from each tail), then take
the top 10.

**Code Task 6.3.4.4**:

In [ ]:
# Calculate trimmed variance (trim 10% from each tail) for all features
# Sort and get the 10 largest, assign to top_ten_trim_var
# Hint: Use df.apply(trimmed_var, limits=(0.1, 0.1))
top_ten_trim_var = (
    df.apply(...).sort_values().tail(10)
)
top_ten_trim_var

Plot the trimmed-variance top 10 as a horizontal bar chart so you can compare it
directly against the raw-variance chart.

**Code 6.3.4.5**:

In [ ]:
# Create horizontal bar chart of trimmed variance
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    orientation="h",
    title="SCF: High Variance Features"
)
fig.update_layout(xaxis_title="Trimmed Variance", yaxis_title="Feature")
fig.show()

📊 Compare the two charts side by side. The trimmed chart has a **smaller scale**
(trimming removed the inflating extremes) and a **reshuffled ranking** —
business-related features that ranked high purely on a few wealthy owners have
receded. This is the ranking you trust for selecting clustering features.

Now extract the names of the top 5 trimmed-variance features — your clustering
feature set.

**Code Task 6.3.4.6**:

In [ ]:
# Extract the column names of the top 5 features by trimmed variance
# Convert to a list and assign to high_var_cols
high_var_cols = top_ten_trim_var.tail(...).index.to_list()
high_var_cols

Build the feature matrix `X` from just those high-variance columns, and confirm
its shape.

**Code 6.3.4.7**:

In [ ]:
# Create feature matrix with selected columns
X = df[high_var_cols]
print("X shape:", X.shape)
X.head()

### Checkpoint

🧪 The asserts confirm you ended with exactly 5 features, that every selected name
really exists in `df`, and that `X` kept all the rows. Passing means your feature
matrix is correctly shaped for clustering.

In [ ]:
assert len(high_var_cols) == 5, (
    f"Expected 5 features, got {len(high_var_cols)}."
)
assert all(col in df.columns for col in high_var_cols), (
    "One or more selected columns not found in DataFrame."
)
assert X.shape[1] == 5, (
    f"Feature matrix should have 5 columns, got {X.shape[1]}."
)
assert X.shape[0] == df.shape[0], (
    f"Feature matrix rows ({X.shape[0]}) should match df rows ({df.shape[0]})."
)
print("Feature selection checkpoint passed!")

## 5. Standardization and Scale Comparison

### Problem

Your five features live on wildly different scales — `ASSET` reaches into the
millions while `WAGEINC` sits in the tens of thousands. Because K-Means uses
Euclidean distance, the large-scale features would dominate. You must standardize
first.

### Approach

First inspect each feature's mean and standard deviation to *quantify* the scale
gap. Then apply `StandardScaler`, and verify the transform produced mean ≈ 0 and
standard deviation ≈ 1 across the board.

### Tasks

Compute the mean and standard deviation of each feature in `X`, cast to integers
so the large numbers are easy to compare.

**Code 6.3.5.1**:

In [ ]:
# Create X_summary with mean and std for all features in X
# Convert to integers for readability using .astype(int)
X_summary = X.agg(["mean", "std"]).astype(int)
X_summary

📊 Read down the std row: the features differ by orders of magnitude. That spread
is the concrete reason standardization isn't optional here — without it, the
single largest-scale feature would effectively *be* the distance metric.

Now instantiate a `StandardScaler`, fit-transform `X`, and store the result in a
new DataFrame `X_scaled` carrying the same column names.

**Code Task 6.3.5.2**:

In [ ]:
# Instantiate a StandardScaler
ss = ...
# Fit and transform X, storing result in X_scaled_data
X_scaled_data = ss.fit_transform(...)
# Create DataFrame X_scaled with same column names as X
X_scaled = pd.DataFrame(X_scaled_data, columns=...)
# Print shape and display first few rows
print("X_scaled shape:", ...)
X_scaled.head()

Verify the transform worked: each feature should now have mean ≈ 0 and standard
deviation ≈ 1.

**Code 6.3.5.3**:

In [ ]:
# Create X_scaled_summary with mean and std for X_scaled
# Round to verify mean ≈ 0 and std ≈ 1
X_scaled_summary = X_scaled.agg(["mean", "std"]).round(2)
X_scaled_summary

✅ Every column now reads mean ≈ 0, std ≈ 1. The orders-of-magnitude gap from two
cells ago is gone — all five features finally speak the same units, and each gets
an equal vote in the distance calculation.

### Checkpoint

🧪 These asserts are stricter than eyeballing: means within `1e-10` of zero and
stds within `0.01` of one. They mathematically confirm the scaler did its job
before you feed the data to K-Means.

In [ ]:
means = X_scaled.mean()
stds = X_scaled.std()
assert all(abs(m) < 1e-10 for m in means), (
    f"Means should be ~0 after standardization, got {means.tolist()}"
)
assert all(abs(s - 1) < 0.01 for s in stds), (
    f"Stds should be ~1 after standardization, got {stds.tolist()}"
)
print("Standardization checkpoint passed!")

## 6. Model Selection with Inertia and Silhouette Score

### Problem

How many clusters best describe this data? Too few oversimplify; too many invent
distinctions that aren't real. You'll let inertia and silhouette score, read
together, settle the question.

### Approach

Train K-Means *pipelines* (scaler + model) for *k* from 2 to 12, recording inertia
and silhouette at each step, then plot both curves to locate the elbow and the
silhouette peak.

📌 Note the metric input: silhouette is computed on the *original* `X` with the
fitted labels, not on the scaled array — a common gotcha.

### Tasks

Loop over *k* = 2 to 12, fitting a fresh `StandardScaler` + `KMeans` pipeline each
time and collecting inertia and silhouette score.

**Code 6.3.6.1**:

In [ ]:
n_clusters = range(2, 13)
inertia_errors = []
silhouette_scores = []

# Train models and collect metrics
for k in n_clusters:
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    model.fit(X)
    inertia_errors.append(model[-1].inertia_)
    silhouette_scores.append(silhouette_score(X, model[-1].labels_))

print("Inertia (first 3):", inertia_errors[:3])
print("Silhouette scores (first 3):", silhouette_scores[:3])

The first few values give you a feel for the trajectory — inertia starting high
and falling, silhouette hovering in its [−1, 1] range. The plots that follow turn
these lists into a decision.

Plot inertia against the number of clusters and look for the **elbow** — where the
steep decline gives way to a gentle one.

**Code 6.3.6.2**:

In [ ]:
# Create a line plot of inertia_errors vs n_clusters
fig = px.line(
    x=n_clusters,
    y=inertia_errors,
    title="K-Means Model: Inertia vs Number of Clusters"
)
# Set x-axis to "Number of Clusters", y-axis to "Inertia"
fig.update_layout(xaxis_title="Number of Clusters", yaxis_title="Inertia")
fig.show()

📊 **Reading the elbow.** The curve drops sharply through the small *k* values,
then flattens. The bend — around 4 clusters here — marks the point of diminishing
returns: past it, extra clusters barely tighten the fit and mostly add complexity.

Now plot silhouette score against the number of clusters; the peak signals the
best-defined clusters.

**Code 6.3.6.3**:

In [ ]:
# Create a line plot of silhouette_scores vs n_clusters
fig = px.line(
    x=n_clusters,
    y=silhouette_scores,
    title="K-Means Model: Silhouette Score vs Number of Clusters"
)
# Set x-axis to "Number of Clusters", y-axis to "Silhouette Score"
fig.update_layout(
    xaxis_title="Number of Clusters",
    yaxis_title="Silhouette Score"
)
fig.show()

📊 **Reading silhouette.** Unlike inertia, this curve can rise and fall, so its
high point is a real recommendation. When the silhouette stays strong at the same
*k* where inertia elbows (3–4 clusters), the two metrics corroborate each other —
giving you confidence rather than a coin flip.

### Checkpoint

🧪 Besides checking you collected 11 values in valid ranges, the last assert
encodes a sanity law of K-Means: inertia at *k*=2 must exceed inertia at *k*=12.
If that fails, something is wrong with the loop or the model.

In [ ]:
assert len(inertia_errors) == 11, (
    f"Expected 11 inertia values (k=2 to 12), got {len(inertia_errors)}."
)
assert len(silhouette_scores) == 11, (
    f"Expected 11 silhouette scores, got {len(silhouette_scores)}."
)
assert all(ie > 0 for ie in inertia_errors), (
    "All inertia values should be positive."
)
assert all(-1 <= ss <= 1 for ss in silhouette_scores), (
    "Silhouette scores should be between -1 and 1."
)
assert inertia_errors[0] > inertia_errors[-1], (
    "Inertia should decrease as k increases."
)
print("Model selection checkpoint passed!")

## 7. Final Model Training

### Problem

The metrics pointed to **4 clusters** as the sweet spot between simplicity and
quality. Now commit to it: train the final model and pull out the per-household
labels for interpretation.

### Approach

Build a `StandardScaler` + `KMeans(n_clusters=4)` pipeline, fit it to `X`, and
extract the labels via `named_steps`.

### Tasks

Create and fit the final 4-cluster pipeline.

**Code Task 6.3.7.1**:

In [ ]:
# Create final_model pipeline with StandardScaler and KMeans
# Use n_clusters=4 and random_state=42
final_model = make_pipeline(
    ...,
    KMeans(n_clusters=..., random_state=...)
)
# Fit the model on X
final_model.fit(...)

Extract the cluster labels through `named_steps`. Each household lands in a group
numbered 0 to 3.

**Code 6.3.7.2**:

In [ ]:
# Extract labels from final_model using named_steps["kmeans"].labels_
labels = final_model.named_steps["kmeans"].labels_
# Print the first 5 labels
print("First 5 labels:", labels[:5])

### Checkpoint

🧪 These asserts verify the pipeline is wired correctly: it exposes `named_steps`,
contains a step literally named `"kmeans"`, was configured for 4 clusters, and
produced one label per household across all four cluster IDs.

In [ ]:
assert hasattr(final_model, "named_steps"), (
    "final_model should be a pipeline with named_steps."
)
assert "kmeans" in final_model.named_steps, (
    "Pipeline should contain a step named 'kmeans'."
)
assert final_model.named_steps["kmeans"].n_clusters == 4, (
    f"Expected 4 clusters, got {final_model.named_steps['kmeans'].n_clusters}."
)
assert len(labels) == X.shape[0], (
    f"Labels length ({len(labels)}) should match X rows ({X.shape[0]})."
)
assert set(labels) == {0, 1, 2, 3}, (
    f"Expected labels 0-3, got unique values: {set(labels)}."
)
print("Final model checkpoint passed!")

## 8. Cluster Interpretation and Visualization

### Problem

Every household now has a cluster label — but a label is just a number until you
explain it. What financial profile does each cluster represent, and how do you
communicate that?

### Approach

Compute the mean of each feature per cluster to build financial profiles,
visualize those profiles with a grouped bar chart, and finally use PCA to render
the four clusters as a 2D scatter plot.

### Tasks

Group `X` by the cluster labels and average each feature within each cluster,
producing a profile table.

**Code 6.3.8.1**:

In [ ]:
# Group X by labels and calculate mean for each cluster
# Assign result to xgb (cluster profile DataFrame)
xgb = X.groupby(labels).mean()
xgb

📊 Each row of this table is a segment; each column is a feature's average for
that segment. Read across a row and you get a one-line description of a household
type — "high assets, low debt" or "moderate assets, high debt." This is where
cluster numbers become business meaning.

Now turn the profile table into a grouped bar chart for easy visual comparison.

**Code 6.3.8.2**:

In [ ]:
# Create grouped bar chart of cluster profiles
fig = px.bar(
    xgb,
    barmode="group",
    title="Mean Household Finances by Cluster"
)
# Set x-axis to "Cluster", y-axis to "Value [$]"
fig.update_layout(xaxis_title="Cluster", yaxis_title="Value [$]")
fig.show()

📊 The grouped bars expose the contrasts at a glance — one cluster might tower in
assets while another spikes in debt. These distinct shapes are what let you name
the segments and, eventually, recommend different products for each.

Now apply PCA to collapse the 5-dimensional feature space to 2 dimensions, stored
as `PC1` and `PC2`.

**Code Task 6.3.8.3**:

In [ ]:
# Instantiate PCA with n_components=2 and random_state=42
pca = PCA(n_components=..., random_state=...)
# Fit and transform X, storing result in X_t
X_t = pca.fit_transform(...)
# Create DataFrame X_pca with columns "PC1" and "PC2"
X_pca = pd.DataFrame(X_t, columns=[...])
# Print shape and display first few rows
print("X_pca shape:", ...)
X_pca.head()

Create a PCA scatter plot colored by cluster label to reveal how separated the
clusters are in the reduced space.

**Code 6.3.8.4**:

In [ ]:
# Create a scatter plot of X_pca with PC1 on x-axis, PC2 on y-axis
# Color points by cluster labels (convert to string for discrete colors)
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()

📊 This scatter is the visual payoff of the whole lesson: a 5D segmentation
flattened into one readable picture. Well-separated color blocks mean the clusters
are genuinely distinct; heavy overlap would warn that the grouping is fragile.
Remember the axes are abstract — PC1 and PC2 are blends of all five original
features, not any single financial variable.

### Checkpoint

🧪 The final asserts confirm the profile table is `(4, 5)` — four clusters by five
features — and that the PCA output has the right shape and the columns `PC1`/`PC2`.
Passing means your interpretation artifacts are correctly built.

In [ ]:
assert xgb.shape == (4, 5), (
    f"Cluster profile shape should be (4, 5), got {xgb.shape}."
)
assert X_pca.shape == (X.shape[0], 2), (
    f"X_pca shape should be ({X.shape[0]}, 2), got {X_pca.shape}."
)
assert list(X_pca.columns) == ["PC1", "PC2"], (
    f"X_pca columns should be ['PC1', 'PC2'], got {list(X_pca.columns)}."
)
print("Visualization checkpoint passed!")

# Wrap-up

In this lesson you accomplished the following:

-   Calculated both regular and trimmed variance to identify informative features
    while accounting for outliers.
-   Selected the top 5 high-variance features for multi-dimensional clustering.
-   Applied standardization so every feature contributes equally to distance
    calculations.
-   Used inertia (elbow method) and silhouette scores to settle on 4 clusters.
-   Built a K-Means pipeline that combines standardization and clustering in one
    workflow.
-   Interpreted cluster profiles by examining mean feature values across clusters.
-   Applied PCA to reduce 5 dimensions to 2 for visualization, producing a scatter
    plot that reveals cluster structure.

🧠 **The bigger picture.** You now have the full multi-feature clustering toolkit:
*choose* features by robust variance, *scale* them so distance is fair, *select* k
with two corroborating metrics, *fit* through a pipeline, and *communicate* through
profiles and PCA. That last word — communicate — is the hinge to what's next.

➡️ **What's next.** A static PCA plot tells the story to *you*. The next notebook
wraps this model in an interactive **Dash** dashboard so a non-technical
stakeholder can explore the segments themselves — choosing features, adjusting the
number of clusters, and seeing the results update live.